In [51]:
# Input take label, latent semantics and k
LABEL_INPUT = input(
"""
Provide the label for which you want to find similar labels
"""
)

LATENT_SEMANTICS = input(
"""
Provide a latent semantic that was generated using tasks 3 to 6.
For eg:
if you generated a latent semantic in task 3 from color space and svd,
enter "LS1_color_svd" if you generated it for tasks where you didn't select the algorithm,
use this format "LS3_color"
If you are unsure what to type here, please see Code/database/<name>_reducer.pt
after running the relevant task. Any <name> can be input here.
"""
)

K = int(input("Enter K, the K most similar labels to find under the latent space."))

In [52]:
from utils.database_utils import retrieve
from utils.distance_utils import top_k_distance_ranker
from scipy.spatial.distance import cityblock, correlation, cosine
from utils.dataset_utils import initialize_dataset

from feature_models.feature_matrix.label_label_similarity import LabelLabelSimilarity
from utils.database_utils import compressed_retrieve

In [53]:
LATENT_SPACE = LATENT_SEMANTICS.split('_')[0]
FEATURE_SPACE = LATENT_SEMANTICS.split('_')[1]

# Load created latent space
reducer = retrieve(f'{LATENT_SEMANTICS}_reducer.pt')

feature_vectors = retrieve(f'{FEATURE_SPACE}.pt')

In [54]:
def find_nearest_cluster(centroids, query_vector, K, distance_fn = cityblock):
    distance = []
    for i, label in enumerate(centroids.keys()):
        distance.append((label, distance_fn(query_vector, centroids[label])))
    distance.sort(key=lambda x:x[1])
    return distance[:K]

In [56]:
if LATENT_SPACE == "LS1":
    # For LS1 find labels for every image in the latent space
    latent_space = reducer.reduce_features(feature_vectors)
    
    centroid_format_latent_space = {}
    for i, feature_item in enumerate(feature_vectors.items()):
        label = feature_item[1][0]
        feature = latent_space[i]
        centroid_format_latent_space[i] = (label, feature)

    # Feed it to the centroid function
    from utils.vector_utils import get_representative_vectors_for_labels
    from utils.dataset_utils import initialize_dataset

    centroids = get_representative_vectors_for_labels(centroid_format_latent_space, initialize_dataset().categories, 1)

    # Find closest labels based on distance
    distances = find_nearest_cluster(centroids, centroids[LABEL_INPUT], K)
    print(distances)
elif LATENT_SPACE == "LS2":
    # For LS2 we pick the matrix from CP decomposition
    labels = initialize_dataset().categories
    latent_space = reducer.label_latent_space_weights
    centroids = {}
    # Make latent space in label format
    for i, label in enumerate(labels):
        centroids[label] = latent_space[i]
    # Find closest labels based on distance
    distances = find_nearest_cluster(centroids, centroids[LABEL_INPUT], K)
    print(distances)
elif LATENT_SPACE == "LS3":
    # For LS3 
    # Find distance directly
    labels = initialize_dataset().categories
    formatter = LabelLabelSimilarity(feature_vectors, labels)
    label_feature_vectors = formatter.get_matrix()
    latent_space = reducer.reduce_features(label_feature_vectors)
    centroids = {}
    # Make latent space in label format
    for i, label in enumerate(labels):
        centroids[label] = latent_space[i]
    # Find closest labels based on distance
    distances = find_nearest_cluster(centroids, centroids[LABEL_INPUT], K)
    print(distances)
else: 
    # For LS4
    image_feature_vectors = compressed_retrieve(f'img_img_{FEATURE_SPACE}.pt')
    latent_space = reducer.reduce_features(image_feature_vectors)
    centroid_format_latent_space = {}
    for i, feature_item in enumerate(feature_vectors.items()):
        label = feature_item[1][0]
        feature = latent_space[i]
        centroid_format_latent_space[i] = (label, feature)

    # Feed it to the centroid function
    from utils.vector_utils import get_representative_vectors_for_labels
    from utils.dataset_utils import initialize_dataset

    centroids = get_representative_vectors_for_labels(centroid_format_latent_space, initialize_dataset().categories, 1)

    # Find closest labels based on distance
    distances = find_nearest_cluster(centroids, centroids[LABEL_INPUT], K)
    print(distances)

Files already downloaded and verified
[('airplanes', 0.0), ('schooner', 1.9857638031952458), ('ferry', 2.050387107476587), ('revolver', 2.109515037289853), ('helicopter', 2.130455484173325), ('ketch', 2.1680601030864146), ('car_side', 2.2928347140143415), ('dalmatian', 2.4236229842873893), ('umbrella', 2.657901695684795), ('Motorbikes', 2.6753538798024667)]


In [ ]:
for index, label_weights in enumerate(distances):
    label = label_weights[0]
    distance = label_weights[1]
    print(str(index + 1) + ".", "\t\tLabel: ", label, "\t\tDistance: ", distance)


1. 		Label:  airplanes 		Distance:  0.0
2. 		Label:  ketch 		Distance:  15.937715250128846
3. 		Label:  schooner 		Distance:  18.15689282015732
4. 		Label:  llama 		Distance:  19.942372315506397
5. 		Label:  helicopter 		Distance:  20.635050931423283
6. 		Label:  ferry 		Distance:  20.77359668104742
7. 		Label:  dalmatian 		Distance:  21.239163806660674
8. 		Label:  hedgehog 		Distance:  21.281636267170192
9. 		Label:  barrel 		Distance:  21.727302509702735
10. 		Label:  flamingo 		Distance:  21.846177185368063


: 

: 

: 